In [1]:
print("hello")

hello


In [2]:
import os
import sqlite3
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# Paths
BASE_DIR = os.path.dirname(os.getcwd())
DB_PATH = os.path.join(BASE_DIR, "data", "credit.db")

# LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# Test DB connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM borrowers")
print(f"✅ DB connected. {cursor.fetchone()[0]} rows in 'borrowers'.")
conn.close()

print("✅ LLM ready")

✅ DB connected. 149717 rows in 'borrowers'.
✅ LLM ready


In [3]:
def get_schema(db_path: str) -> str:
    """Get the database schema as a formatted string for the LLM."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    
    schema_parts = []
    for (table_name,) in tables:
        # Get column info
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        
        col_descriptions = []
        for col in columns:
            col_name = col[1]
            col_type = col[2]
            col_descriptions.append(f"  {col_name} ({col_type})")
        
        schema_parts.append(f"Table: {table_name}\n" + "\n".join(col_descriptions))
    
    conn.close()
    return "\n\n".join(schema_parts)


# Get and print schema
schema = get_schema(DB_PATH)
print(schema)

Table: borrowers
  borrower_id (INTEGER)
  is_default (INTEGER)
  credit_utilization (REAL)
  age (INTEGER)
  past_due_30_59 (INTEGER)
  debt_ratio (REAL)
  monthly_income (REAL)
  open_credit_lines (INTEGER)
  past_due_90 (INTEGER)
  real_estate_loans (INTEGER)
  past_due_60_89 (INTEGER)
  num_dependents (INTEGER)
  income_was_missing (INTEGER)


In [4]:
SCHEMA_DESCRIPTION = """
Table: borrowers (149,717 rows of credit risk data)

Columns:
  borrower_id (INTEGER) - unique ID for each borrower
  is_default (INTEGER) - 1 if borrower defaulted within 2 years, 0 otherwise
  credit_utilization (REAL) - ratio of credit used vs available (0 to 1+)
  age (INTEGER) - age of borrower in years
  past_due_30_59 (INTEGER) - number of times 30-59 days past due
  debt_ratio (REAL) - monthly debt payments / monthly income
  monthly_income (REAL) - monthly income in dollars
  open_credit_lines (INTEGER) - number of open loans/credit lines
  past_due_90 (INTEGER) - number of times 90+ days past due
  real_estate_loans (INTEGER) - number of mortgage/real estate loans
  past_due_60_89 (INTEGER) - number of times 60-89 days past due
  num_dependents (INTEGER) - number of dependents
  income_was_missing (INTEGER) - 1 if income data was originally missing, 0 otherwise
"""

print(SCHEMA_DESCRIPTION)


Table: borrowers (149,717 rows of credit risk data)

Columns:
  borrower_id (INTEGER) - unique ID for each borrower
  is_default (INTEGER) - 1 if borrower defaulted within 2 years, 0 otherwise
  credit_utilization (REAL) - ratio of credit used vs available (0 to 1+)
  age (INTEGER) - age of borrower in years
  past_due_30_59 (INTEGER) - number of times 30-59 days past due
  debt_ratio (REAL) - monthly debt payments / monthly income
  monthly_income (REAL) - monthly income in dollars
  open_credit_lines (INTEGER) - number of open loans/credit lines
  past_due_90 (INTEGER) - number of times 90+ days past due
  real_estate_loans (INTEGER) - number of mortgage/real estate loans
  past_due_60_89 (INTEGER) - number of times 60-89 days past due
  num_dependents (INTEGER) - number of dependents
  income_was_missing (INTEGER) - 1 if income data was originally missing, 0 otherwise



In [5]:
from langchain_core.tools import tool

@tool
def sql_query(query: str) -> str:
    """
    Execute a read-only SQL SELECT query on the credit database and return the results.
    Use this to answer questions about borrowers, defaults, income, age, and credit risk.
    Only SELECT queries are allowed. The table is named 'borrowers'.
    """
    # Safety: block non-SELECT queries
    cleaned = query.strip().lower()
    if not cleaned.startswith("select"):
        return "Error: Only SELECT queries are allowed."
    
    # Block dangerous keywords
    forbidden = ["drop", "delete", "update", "insert", "alter", "create"]
    if any(word in cleaned for word in forbidden):
        return "Error: Query contains forbidden keywords."
    
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()
        
        # Get column names
        col_names = [desc[0] for desc in cursor.description]
        
        conn.close()
        
        # Limit output to avoid token overflow
        if len(rows) > 50:
            preview = rows[:50]
            return f"Columns: {col_names}\nRows (showing 50 of {len(rows)}): {preview}"
        
        return f"Columns: {col_names}\nRows: {rows}"
    
    except Exception as e:
        return f"Error executing query: {str(e)}"


print("✅ sql_query tool created")

✅ sql_query tool created


In [7]:
# Test 1: Valid query
print("=== Test 1: Count defaults ===")
result = sql_query.invoke({"query": "SELECT COUNT(*) FROM borrowers WHERE is_default = 1"})
print(result)

# Test 2: Multiple columns
print("\n=== Test 2: Top 5 oldest ===")
result = sql_query.invoke({"query": "SELECT borrower_id, age FROM borrowers ORDER BY age DESC LIMIT 5"})
print(result)

# Test 3: SECURITY — try to DROP (should be blocked)
print("\n=== Test 3: Security — DROP attempt ===")
result = sql_query.invoke({"query": "DROP TABLE borrowers"})
print(result)

# Test 4: SECURITY — sneaky DELETE (should be blocked)
print("\n=== Test 4: Security — DELETE attempt ===")
result = sql_query.invoke({"query": "SELECT * FROM borrowers; DELETE FROM borrowers"})
print(result)

# Test 5: Bad query (wrong column — should return error, not crash)
print("\n=== Test 5: Wrong column name ===")
result = sql_query.invoke({"query": "SELECT nonexistent_column FROM borrowers"})
print(result)

=== Test 1: Count defaults ===
Columns: ['COUNT(*)']
Rows: [(9878,)]

=== Test 2: Top 5 oldest ===
Columns: ['borrower_id', 'age']
Rows: [(2921, 99), (23963, 99), (39773, 99), (57411, 99), (59275, 99)]

=== Test 3: Security — DROP attempt ===
Error: Only SELECT queries are allowed.

=== Test 4: Security — DELETE attempt ===
Error: Query contains forbidden keywords.

=== Test 5: Wrong column name ===
Error executing query: no such column: nonexistent_column


In [13]:
from langchain_core.messages import SystemMessage

# Bind tool to LLM
llm_with_sql = llm.bind_tools([sql_query])

# System prompt with schema
SYSTEM_PROMPT = f"""You are a helpful data analyst assistant for a credit risk database.

You have access to a SQL database with the following schema:

{SCHEMA_DESCRIPTION}

When the user asks a question:
1. Write a SQL SELECT query to get the data
2. Use the sql_query tool to execute it
3. Interpret the results and give a clear, natural language answer

IMPORTANT RULES:
- Only use SELECT queries
- Always use the exact column names from the schema
- For percentages, calculate them in SQL using ROUND(100.0 * ... , 2)

HANDLING "AGE GROUPS":
When a user asks about "age groups" or "age brackets", DO NOT group by individual age.
Instead, create buckets using CASE WHEN:
  CASE 
    WHEN age < 30 THEN '<30'
    WHEN age < 45 THEN '30-44'
    WHEN age < 60 THEN '45-59'
    ELSE '60+'
  END

STATISTICAL SIGNIFICANCE:
When finding "highest" or "lowest" rates, avoid groups with very few samples.
Add a minimum count filter like HAVING COUNT(*) > 100 to avoid misleading outliers.

EXAMPLE:
Question: "Which age group has the highest default rate?"
SQL:
  SELECT 
    CASE 
      WHEN age < 30 THEN '<30'
      WHEN age < 45 THEN '30-44'
      WHEN age < 60 THEN '45-59'
      ELSE '60+'
    END as age_group,
    COUNT(*) as total,
    ROUND(100.0 * SUM(is_default) / COUNT(*), 2) as default_rate_pct
  FROM borrowers
  GROUP BY age_group
  HAVING COUNT(*) > 100
  ORDER BY default_rate_pct DESC
"""

print("✅ Improved system prompt")
print(f"Length: {len(SYSTEM_PROMPT)} chars")

✅ Improved system prompt
Length: 2244 chars


In [14]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage


# State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# Node 1: LLM
def call_llm(state: AgentState):
    response = llm_with_sql.invoke(state["messages"])
    return {"messages": [response]}


# Node 2: Tools
def call_tools(state: AgentState):
    last_message = state["messages"][-1]
    tool_messages = []
    
    for tool_call in last_message.tool_calls:
        print(f"🔧 SQL: {tool_call['args']['query']}")
        result = sql_query.invoke(tool_call["args"])
        tool_messages.append(ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        ))
    
    return {"messages": tool_messages}


# Conditional
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"


# Build graph
builder = StateGraph(AgentState)
builder.add_node("llm", call_llm)
builder.add_node("tools", call_tools)

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "llm")

sql_agent = builder.compile()

print("✅ SQL Agent compiled")



✅ SQL Agent compiled


In [15]:
def ask(question: str):
    """Ask the SQL agent a question and print the answer."""
    print(f"❓ Question: {question}\n")
    
    result = sql_agent.invoke({
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=question)
        ]
    })
    
    answer = result["messages"][-1].content
    print(f"\n💬 Answer: {answer}")
    print("=" * 60)
    return answer


# First question
ask("How many borrowers defaulted on their loans?")

❓ Question: How many borrowers defaulted on their loans?

🔧 SQL: SELECT COUNT(*) as defaulted_count FROM borrowers WHERE is_default = 1

💬 Answer: There are 9,878 borrowers who defaulted on their loans.


'There are 9,878 borrowers who defaulted on their loans.'

In [16]:
# Question 2: Percentage
ask("What percentage of borrowers defaulted?")

# Question 3: Aggregation with condition
ask("What is the average monthly income of borrowers who defaulted vs those who didn't?")

# Question 4: Complex — age groups
ask("Which age group has the highest default rate?")

# Question 5: Top N
ask("Show me the 5 borrowers with the highest credit utilization.")

❓ Question: What percentage of borrowers defaulted?

🔧 SQL: SELECT ROUND(100.0 * SUM(is_default) / COUNT(*), 2) as default_rate_pct FROM borrowers

💬 Answer: Approximately 6.6% of borrowers in the database defaulted.
❓ Question: What is the average monthly income of borrowers who defaulted vs those who didn't?

🔧 SQL: SELECT is_default, AVG(monthly_income) as avg_monthly_income FROM borrowers GROUP BY is_default

💬 Answer: The average monthly income of borrowers who did not default is approximately $6,480, while the average monthly income of borrowers who defaulted is around $5,621.
❓ Question: Which age group has the highest default rate?

🔧 SQL: SELECT 
  CASE 
    WHEN age < 30 THEN '<30'
    WHEN age < 45 THEN '30-44'
    WHEN age < 60 THEN '45-59'
    ELSE '60+'
  END as age_group,
  COUNT(*) as total,
  ROUND(100.0 * SUM(is_default) / COUNT(*), 2) as default_rate_pct
FROM borrowers
GROUP BY age_group
HAVING COUNT(*) > 100
ORDER BY default_rate_pct DESC

💬 Answer: The age group wi

'The 5 borrowers with the highest credit utilization are as follows:\n\n1. Borrower ID: 192\n   - Credit Utilization: 1.093\n   - Age: 53\n   - Monthly Income: $3500\n   - Other relevant data available\n\n2. Borrower ID: 227\n   - Credit Utilization: 1.093\n   - Age: 38\n   - Monthly Income: $3556\n   - Other relevant data available\n\n3. Borrower ID: 294\n   - Credit Utilization: 1.093\n   - Age: 45\n   - Monthly Income: $8333\n   - Other relevant data available\n\n4. Borrower ID: 542\n   - Credit Utilization: 1.093\n   - Age: 24\n   - Monthly Income: $5400\n   - Other relevant data available\n\n5. Borrower ID: 668\n   - Credit Utilization: 1.093\n   - Age: 33\n   - Monthly Income: $2700\n   - Other relevant data available'